## Read Data

In [8]:
!pip install fastparquet

  Obtaining dependency information for fastparquet from https://files.pythonhosted.org/packages/27/13/abd53c73d1a146ffae523285214c3db3dafe855bd70af787bf9bf9295224/fastparquet-2025.12.0-cp311-cp311-macosx_10_9_universal2.whl.metadata
  Obtaining dependency information for cramjam>=2.3 from https://files.pythonhosted.org/packages/0b/f3/001d00070ca92e5fbe6aacc768e455568b0cde46b0eb944561a4ea132300/cramjam-2.11.0-cp311-cp311-macosx_10_12_x86_64.whl.metadata
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 891.1/891.1 kB 7.4 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 22.6 MB/s eta 0:00:0000:0100:01


In [117]:
import pandas as pd

ratings = pd.read_parquet("data/ratings-20240501-20240531.parquet")
notes = pd.read_parquet("data/notes-20240501-20240531.parquet")
political = pd.read_csv("data/database_replication.csv")


## Data Preprocess

In [118]:
# Rename political columns to match
political = political.rename(columns={'note_id': 'noteId', 'tweet_id': 'tweetId'})

# Join notes (ratings events) with political on noteId
notes_political = notes.merge(political[['noteId', 'tweetId', 'party', 'topic']], on='noteId', how='left')

print("Joined shape:", notes_political.shape)
print("Party value counts:", notes_political['party'].value_counts())
print("Null party %:", notes_political['party'].isna().mean().round(3))

Joined shape: (6385001, 15)
Party value counts: party
republican    1099443
democrat       834051
unknown        706433
Name: count, dtype: int64
Null party %: 0.587


### Paritsanship Signals

In [119]:
# Initialize author_signals dataframe
author_signals = pd.DataFrame(index=notes_political['noteAuthorParticipantId'].dropna().unique())
print(author_signals.shape)

(45530, 0)


In [120]:
# Average party for authors of tweets they wrote notes about (encode republican as -1 and democrat as 1).
import numpy as np

party_map = {'republican': -1, 'democrat': 1, 'unknown': np.nan}
notes_political['party_num'] = notes_political['party'].map(party_map)

author_groups = notes_political.groupby('noteAuthorParticipantId')
avg_party_wrote = author_groups['party_num'].mean()

In [121]:
# Average party for authors of tweets where they rated the note helpful
helpful_mask = notes_political['helpfulnessLevel'].isin(['HELPFUL', 'SOMEWHAT_HELPFUL'])
avg_party_rated_helpful = notes_political[helpful_mask].groupby('raterParticipantId')['party_num'].mean()

# Average party for authors of tweets where they rated the note not helpful
not_helpful_mask = notes_political['helpfulnessLevel'].isin(['NOT_HELPFUL'])
avg_party_rated_not_helpful = notes_political[not_helpful_mask].groupby('raterParticipantId')['party_num'].mean()

# print("Average party for helpful tweets:")
# print(avg_party_rated_helpful.head())
# print("\nAverage party for not helpful tweets:")
# print(avg_party_rated_not_helpful.head())

In [122]:
# Average note factor of notes they rated helpful
avg_factor_rated_helpful = notes_political[helpful_mask].groupby('raterParticipantId')['noteFinalFactor'].mean()

# Average note factor of notes they rated not helpful
avg_factor_rated_not_helpful = notes_political[not_helpful_mask].groupby('raterParticipantId')['noteFinalFactor'].mean()

In [123]:
# Percent of authors of tweets they wrote notes about that were republican
notes_political['is_republican'] = notes_political['party'] == 'republican'
notes_political['is_republican'] = notes_political['is_republican'].where(
    notes_political['party'].isin(['republican', 'democrat']), np.nan
)

percent_republican_wrote = author_groups['is_republican'].mean()

In [124]:
# Percent of authors of tweets that were republican for posts they rated helpful
percent_republican_helpful = notes_political[helpful_mask].groupby('raterParticipantId')['is_republican'].mean()

# Percent of authors of tweets that were republican for posts they rated not helpful
percent_republican_not_helpful = notes_political[not_helpful_mask].groupby('raterParticipantId')['is_republican'].mean()

In [125]:
from scipy.stats import entropy

# Number of unique topics they wrote notes about
num_unique_topics_wrote = author_groups['topic'].nunique()

# Number of unique topics rated
rater_groups = notes_political.groupby('raterParticipantId')
num_unique_topics_rated = rater_groups['topic'].nunique()   

In [126]:
# Shannon entropy for topics of posts they rated
def shannon_entropy(series):
    counts = series.value_counts()
    return entropy(counts, base=2)

topic_entropy_rated = rater_groups['topic'].apply(shannon_entropy)

# Shannon entropy for topics of posts they wrote notes for
topic_entropy_wrote = author_groups['topic'].apply(shannon_entropy)

### Skill Signals

In [127]:
# Average note intercept for notes they wrote
note_intercept_wrote = author_groups['noteFinalIntercept'].mean()

# Percent of notes they wrote that earned "helpful" rating
ratings['is_helpful'] = ratings['noteFinalRatingStatus'] == 'CURRENTLY_RATED_HELPFUL'
percent_notes_helpful = ratings.groupby('noteAuthorParticipantId')['is_helpful'].mean()

# Percent of ratings they gave where their rating agreed with the helpfulness the post went on to earn
rated_helpful = notes_political['helpfulnessLevel'].isin(['HELPFUL', 'SOMEWHAT_HELPFUL'])
rated_not_helpful = notes_political['helpfulnessLevel'] == 'NOT_HELPFUL'
final_helpful = notes_political['noteFinalRatingStatus'].isin(['CURRENTLY_RATED_HELPFUL'])
final_not_helpful = notes_political['noteFinalRatingStatus'] == 'CURRENTLY_RATED_NOT_HELPFUL'

rating_agreed = ((rated_helpful & final_helpful) | (rated_not_helpful & final_not_helpful)).groupby(notes_political['raterParticipantId']).mean()

### Comebine

In [128]:
# author
author_signals = pd.DataFrame({
    'avg_party_wrote': avg_party_wrote,
    'percent_republican_wrote': percent_republican_wrote,
    'num_unique_topics_wrote': num_unique_topics_wrote,
    'topic_entropy_wrote': topic_entropy_wrote,
    'avg_note_intercept_wrote': note_intercept_wrote,
    'percent_notes_helpful': percent_notes_helpful
})


# rater
rater_signals = pd.DataFrame({
    'avg_party_rated_helpful': avg_party_rated_helpful,
    'avg_party_rated_not_helpful': avg_party_rated_not_helpful,
    'avg_factor_rated_helpful': avg_factor_rated_helpful,
    'avg_factor_rated_not_helpful': avg_factor_rated_not_helpful,
    'percent_republican_helpful': percent_republican_helpful,
    'percent_republican_not_helpful': percent_republican_not_helpful,
    'num_unique_topics_rated': num_unique_topics_rated,
    'topic_entropy_rated': topic_entropy_rated,
    'rating_agreed': rating_agreed,
})

author_signals.index.name = 'userID'
rater_signals.index.name = 'userID'
author_signals = author_signals.reset_index()
rater_signals = rater_signals.reset_index()

In [129]:
user_signals = author_signals.merge(rater_signals, on='userID', how='outer')

In [130]:
print(user_signals.info())
print(user_signals.head())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 502092 entries, 0 to 502091
Data columns (total 16 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   userID                          502092 non-null  object 
 1   avg_party_wrote                 14917 non-null   float64
 2   percent_republican_wrote        14917 non-null   object 
 3   num_unique_topics_wrote         45530 non-null   float64
 4   topic_entropy_wrote             45530 non-null   float64
 5   avg_note_intercept_wrote        41885 non-null   float64
 6   percent_notes_helpful           45530 non-null   float64
 7   avg_party_rated_helpful         218219 non-null  float64
 8   avg_party_rated_not_helpful     140993 non-null  float64
 9   avg_factor_rated_helpful        447357 non-null  float64
 10  avg_factor_rated_not_helpful    275440 non-null  float64
 11  percent_republican_helpful      218219 non-null  object 
 12  percent_republic

In [131]:
user_signals.info(0)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 502092 entries, 0 to 502091
Data columns (total 16 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   userID                          502092 non-null  object 
 1   avg_party_wrote                 14917 non-null   float64
 2   percent_republican_wrote        14917 non-null   object 
 3   num_unique_topics_wrote         45530 non-null   float64
 4   topic_entropy_wrote             45530 non-null   float64
 5   avg_note_intercept_wrote        41885 non-null   float64
 6   percent_notes_helpful           45530 non-null   float64
 7   avg_party_rated_helpful         218219 non-null  float64
 8   avg_party_rated_not_helpful     140993 non-null  float64
 9   avg_factor_rated_helpful        447357 non-null  float64
 10  avg_factor_rated_not_helpful    275440 non-null  float64
 11  percent_republican_helpful      218219 non-null  object 
 12  percent_republic

## Data Analysis

In [132]:
# Only rows where we know the party
known_party = notes_political[notes_political['party'].isin(['republican', 'democrat'])].copy()

# Correlate party_num with note factor
corr = known_party[['party_num', 'noteFinalFactor']].corr()

In [77]:
rating_counts = notes_political.groupby('raterParticipantId').size()
active_raters = rating_counts[rating_counts > 1].index
user_signals = user_signals[user_signals['userID'].isin(active_raters)]

print("Filtered shape:", user_signals.shape)

Filtered shape: (350091, 16)


In [78]:
notes_political_filtered = notes_political[notes_political['raterParticipantId'].isin(active_raters)]

# Now correlate
known_party = notes_political_filtered[notes_political_filtered['party'].isin(['republican', 'democrat'])].copy()
corr = known_party[['party_num', 'noteFinalFactor']].corr()
print(corr)
print("\nN rows used:", len(known_party))

                 party_num  noteFinalFactor
party_num         1.000000         0.575144
noteFinalFactor   0.575144         1.000000

N rows used: 1891156


In [109]:
rating_counts[rating_counts > 15].sort_values(ascending=False)

raterParticipantId
6646EFB754964DA906B219128732E23A2C4E571BF5A0E47CAC0592ADC6B36793    5323
7A2D56502C5AA9B330C429DA861934DE400E079C81CB02065F3ED2560CE268C6    4903
C0DA0E1225995F1773D4CDF8D1BD8BB52A242F9478D0896DFDF73F8ED431A17A    4200
7D96B24678C0C0CA0573A9736BEFD4638D385327A09F7F52F80230C3EE68CFA2    4190
6F8D01FAFA94F6F5B60F1D81D5CD23AAFFC7683B8DA2810B575074B4F2F07A07    4132
                                                                    ... 
A93D03AAC2BAD97188960C6CE893D9421A620873978C2B49CDAEA7DB91401AC8      16
75ED6814CC8BD5F3204E4FAF475E04C8BAFCE8F6BA8685AC2C84DFE0A5F4999E      16
A93D4E626A65C3CFE2BBAF94898D18E0684B2EF89097CA75DBE327B359B22380      16
C92139FF3C896E8D38771670E6AAB8701970339184DD1BC04086ACB4451418FD      16
5D511F9367163EDF69D032DA98CE6816F9C329719E61110AF6B0F7632B46413C      16
Length: 91874, dtype: int64